In [ ]:
%matplotlib widget

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from ipywidgets import VBox, HBox, Layout, HTML
from IPython.display import display

plt.ioff()

# ==============================================================================
# USAGE
#
# BUTTERWORTH LOW-PASS FILTER DESIGN EXERCISE
#
# Specifications:
#
#       ωp = 0.6 rad/s
#       ωs = 2.5 rad/s
#       Ap = 1.0 dB
#       As = 40.0 dB
#
# The minimum order is calculated numerically from the theoretical formula.
#
# After N has been determined, all remaining quantities are constructed
# symbolically with SymPy:
#
#       Butterworth poles
#       conjugate-pole factors
#       H(s)
#       H(jω)
#       |H(jω)|
#       phase response
#       group delay
#
# No coefficients from the final analytical solution are hard-coded.
# ==============================================================================

# ==============================================================================
# JUPYTER DISPLAY SETTINGS
# ==============================================================================

display(HTML("""
<style>

.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.output,
.output_area,
.output_subarea,
.output_scroll {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.jupyter-widgets,
.widget-box,
.widget-html,
.widget-html-content {
    overflow: visible !important;
    max-height: none !important;
}

.jp-Cell-outputWrapper {
    overflow: visible !important;
}

</style>
"""))

# ==============================================================================
# FILTER SPECIFICATIONS
# ==============================================================================

wp = 0.6
ws = 2.5
Ap = 1.0
As = 40.0

# ==============================================================================
# STEP 1: MINIMUM FILTER ORDER
# ==============================================================================

ratio_A = (10.0**(As / 10.0) - 1.0) / (10.0**(Ap / 10.0) - 1.0)
ratio_w = ws / wp

N_exact = 0.5 * np.log10(ratio_A) / np.log10(ratio_w)
N = int(np.ceil(N_exact))

# ==============================================================================
# SYMBOLIC VARIABLES
# ==============================================================================

s = sp.symbols('s', real=True)
omega = sp.symbols('omega', real=True)
I = sp.I

# ==============================================================================
# AUXILIARY SYMBOLIC FUNCTION
# ==============================================================================

def simplify_polynomial(expr, variable):

    polynomial = sp.Poly(sp.expand(expr), variable)

    result = sp.Integer(0)

    for powers, coefficient in polynomial.terms():

        degree = powers[0]

        coefficient = sp.simplify(coefficient)
        coefficient = sp.trigsimp(coefficient)
        coefficient = sp.radsimp(coefficient)

        result += coefficient * variable**degree

    return sp.expand(result)

# ==============================================================================
# STEP 2: SYMBOLIC NORMALIZED BUTTERWORTH POLES
# ==============================================================================

prototype_poles_symbolic = []

for k in range(N):

    theta_k = sp.pi / 2 + (2 * k + 1) * sp.pi / (2 * N)

    p_k = sp.cos(theta_k) + I * sp.sin(theta_k)

    p_k = sp.simplify(p_k)

    prototype_poles_symbolic.append(p_k)

# ==============================================================================
# STEP 3: SYMBOLIC DENOMINATOR
#
# D(s) = Π(s - pk)
# ==============================================================================

denominator_symbolic = sp.Integer(1)

for p_k in prototype_poles_symbolic:

    denominator_symbolic = sp.expand(denominator_symbolic * (s - p_k))

denominator_symbolic = simplify_polynomial(denominator_symbolic, s)

# DC normalization
numerator_symbolic = sp.simplify(denominator_symbolic.subs(s, 0))

H_s = sp.simplify(numerator_symbolic / denominator_symbolic)

# ==============================================================================
# STEP 4: CONJUGATE-POLE SECOND-ORDER FACTORS
# ==============================================================================

upper_half_plane_poles = []

for p_k in prototype_poles_symbolic:

    if float(sp.N(sp.im(p_k))) > 0.0:

        upper_half_plane_poles.append(p_k)

quadratic_factors = []

for p_k in upper_half_plane_poles:

    factor_k = sp.expand((s - p_k) * (s - sp.conjugate(p_k)))

    factor_k = simplify_polynomial(factor_k, s)

    quadratic_factors.append(factor_k)

# ==============================================================================
# STEP 5: SYMBOLIC FREQUENCY RESPONSE
#
# s -> jω
# ==============================================================================

denominator_jw = sp.expand(denominator_symbolic.subs(s, I * omega))

den_real = sp.simplify(sp.re(denominator_jw))
den_imag = sp.simplify(sp.im(denominator_jw))

den_real = simplify_polynomial(den_real, omega)
den_imag = simplify_polynomial(den_imag, omega)

H_jw = sp.simplify(numerator_symbolic / denominator_jw)

# ==============================================================================
# STEP 6: SYMBOLIC MAGNITUDE RESPONSE
# ==============================================================================

magnitude_denominator = sp.expand(den_real**2 + den_imag**2)
magnitude_denominator = simplify_polynomial(magnitude_denominator, omega)

magnitude_squared_symbolic = sp.simplify(numerator_symbolic**2 / magnitude_denominator)

magnitude_symbolic = sp.sqrt(magnitude_squared_symbolic)

# ==============================================================================
# STEP 7: SYMBOLIC PHASE RESPONSE
# ==============================================================================

phase_ratio_symbolic = sp.simplify(den_imag / den_real)

# ==============================================================================
# STEP 8: SYMBOLIC GROUP DELAY
#
# D(jω) = R(ω) + j I(ω)
#
# τ(ω) = [R I' - I R'] / [R² + I²]
# ==============================================================================

den_real_derivative = sp.diff(den_real, omega)
den_imag_derivative = sp.diff(den_imag, omega)

group_delay_numerator = sp.expand(den_real * den_imag_derivative - den_imag * den_real_derivative)

group_delay_denominator = sp.expand(den_real**2 + den_imag**2)

group_delay_numerator = simplify_polynomial(group_delay_numerator, omega)
group_delay_denominator = simplify_polynomial(group_delay_denominator, omega)

group_delay_symbolic = sp.cancel(group_delay_numerator / group_delay_denominator)
group_delay_symbolic = sp.factor(group_delay_symbolic)

# ==============================================================================
# NUMERICAL FUNCTIONS FOR PLOTTING ONLY
# ==============================================================================

magnitude_function = sp.lambdify(omega, magnitude_symbolic, modules='numpy')

den_real_function = sp.lambdify(omega, den_real, modules='numpy')

den_imag_function = sp.lambdify(omega, den_imag, modules='numpy')

group_delay_function = sp.lambdify(omega, group_delay_symbolic, modules='numpy')

# ==============================================================================
# FREQUENCY AXIS
# ==============================================================================

omega_values = np.logspace(-2, 2, 5000)

# ==============================================================================
# NUMERICAL MAGNITUDE VALUES
# ==============================================================================

magnitude_values = np.asarray(magnitude_function(omega_values), dtype=float)

# ==============================================================================
# NUMERICAL PHASE VALUES
# ==============================================================================

den_real_values = np.asarray(den_real_function(omega_values), dtype=float)

den_imag_values = np.asarray(den_imag_function(omega_values), dtype=float)

phase_values = -np.unwrap(np.arctan2(den_imag_values, den_real_values))

phase_deg_values = np.rad2deg(phase_values)

# ==============================================================================
# NUMERICAL GROUP-DELAY VALUES
# ==============================================================================

group_delay_values = np.asarray(group_delay_function(omega_values), dtype=float)

if group_delay_values.ndim == 0:

    group_delay_values = np.full_like(omega_values, float(group_delay_values))

# ==============================================================================
# NUMERICAL POLES FOR DISPLAY ONLY
# ==============================================================================

prototype_poles_numeric = []

for p_k in prototype_poles_symbolic:

    prototype_poles_numeric.append(complex(sp.N(p_k, 12)))

prototype_poles_numeric = sorted(prototype_poles_numeric, key=lambda z: (-z.imag, z.real))

pole_text = '<br>'.join([f'p{k + 1} = {p.real:+.6f} {p.imag:+.6f}j' for k, p in enumerate(prototype_poles_numeric)])

# ==============================================================================
# SECOND-ORDER FACTORS FOR DISPLAY
# ==============================================================================

factor_text = ''

for index, factor_k in enumerate(quadratic_factors):

    factor_numeric = sp.N(factor_k, 8)

    factor_text += f'Pair {index + 1}: <span style="color:#0066cc;">{sp.sstr(factor_numeric)}</span><br>'

# ==============================================================================
# NUMERICAL FORMS OF SYMBOLIC EXPRESSIONS FOR DISPLAY
# ==============================================================================

denominator_numeric = sp.N(denominator_symbolic, 8)

den_real_numeric = sp.N(den_real, 8)

den_imag_numeric = sp.N(den_imag, 8)

magnitude_numeric = sp.N(magnitude_symbolic, 8)

phase_ratio_numeric = sp.N(phase_ratio_symbolic, 8)

group_delay_numerator_numeric = sp.N(group_delay_numerator, 8)

group_delay_denominator_numeric = sp.N(group_delay_denominator, 8)

# ==============================================================================
# DESCRIPTION
# ==============================================================================

description = HTML(f"""
<div style="
    border:1px solid #9ec9f5;
    border-radius:7px;
    padding:9px 11px;
    margin:0px 0px 8px 0px;
    font-size:12px;
    line-height:1.50;
    background-color:#f7fbff;
    width:1240px;
    max-width:1240px;
    box-sizing:border-box;
">
<b>Butterworth Design Exercise</b><br>
Construct a normalized Butterworth filter with
ω<sub>p</sub> = {wp:.1f} rad/s,
ω<sub>s</sub> = {ws:.1f} rad/s,
A<sub>p</sub> = {Ap:.1f} dB and
A<sub>s</sub> = {As:.0f} dB.
<br>
<b>Purpose:</b>
Reproduce the analytical solution symbolically and visualize the resulting
magnitude response, phase response and group delay. After the filter order is
determined from the theoretical design equation, all remaining expressions are
constructed symbolically with SymPy.
</div>
""", layout=Layout(width='1250px', max_width='1250px'))

# ==============================================================================
# LEFT COLUMN: SYMBOLIC / NUMERICAL SOLUTION
# ==============================================================================

info_html = HTML(f"""
<div style="
    border:1px solid #cccccc;
    border-radius:7px;
    padding:10px 11px;
    font-size:12px;
    line-height:1.58;
    background:white;
    width:530px;
    box-sizing:border-box;
">

<div style="font-size:12px; font-weight:bold;">
Step 1 — Filter specifications
</div>

<div style="color:#0066cc; margin-top:3px;">
ωp = {wp:.1f} rad/s<br>
ωs = {ws:.1f} rad/s<br>
Ap = {Ap:.1f} dB<br>
As = {As:.0f} dB
</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">

<div style="font-size:12px; font-weight:bold;">
Step 2 — Minimum filter order
</div>

<div style="margin-top:3px;">
N<sub>min</sub> =
<span style="color:#0066cc;">{N_exact:.6f}</span><br>
N =
<span style="color:#0066cc;"><b>{N}</b></span>
</div>

</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">

<div style="font-size:12px; font-weight:bold;">
Step 3 — Symbolically generated stable poles
</div>

<div style="color:#0066cc; margin-top:3px;">
{pole_text}
</div>

</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">

<div style="font-size:12px; font-weight:bold;">
Step 4 — Symbolically generated pole-pair factors
</div>

<div style="margin-top:4px;">
{factor_text}
</div>

</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">

<div style="font-size:12px; font-weight:bold;">
Step 5 — Transfer function
</div>

<div style="margin-top:4px;">
H(s) = 1 / D(s)
</div>

<div style="color:#0066cc; margin-top:3px;">
D(s) = {sp.sstr(denominator_numeric)}
</div>

</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">

<div style="font-size:12px; font-weight:bold;">
Step 6 — Frequency response denominator
</div>

<div style="margin-top:4px;">
D(jω) =
<span style="color:#0066cc;">
({sp.sstr(den_real_numeric)}) + j({sp.sstr(den_imag_numeric)})
</span>
</div>

</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">

<div style="font-size:12px; font-weight:bold;">
Step 7 — Magnitude response
</div>

<div style="color:#0066cc; margin-top:4px;">
|H(jω)| = {sp.sstr(magnitude_numeric)}
</div>

</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">

<div style="font-size:12px; font-weight:bold;">
Step 8 — Phase response
</div>

<div style="margin-top:4px;">
∠H(jω) = −tan⁻¹[
<span style="color:#0066cc;">{sp.sstr(phase_ratio_numeric)}</span>
]
</div>

</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">

<div style="font-size:12px; font-weight:bold;">
Step 9 — Group delay
</div>

<div style="margin-top:4px;">
τ(ω) =
<span style="color:#0066cc;">
({sp.sstr(group_delay_numerator_numeric)}) /
({sp.sstr(group_delay_denominator_numeric)})
</span>
</div>

<div style="margin-top:5px; font-size:11px;">
The complete group-delay expression is generated symbolically from D(jω).
No final coefficients are supplied to the calculation.
</div>

</div>

</div>
""", layout=Layout(width='540px', max_width='540px'))

# ==============================================================================
# COMMON FIGURE SETTINGS
# ==============================================================================

title_fontsize = 11
label_fontsize = 9
tick_fontsize = 8
legend_fontsize = 8

# ==============================================================================
# FIGURE 1: MAGNITUDE RESPONSE
# ==============================================================================

fig_mag, ax_mag = plt.subplots(figsize=(5.3, 3.0))

mag_line, = ax_mag.plot(omega_values, magnitude_values, 'r-', linewidth=2.0, label='|H(jω)|')

ax_mag.axvline(1.0, color='black', linestyle=':', linewidth=1.0, label='ω = 1')
ax_mag.axhline(1.0 / np.sqrt(2.0), color='gray', linestyle='--', linewidth=0.9, label='1/√2')

ax_mag.set_xscale('log')
ax_mag.set_xlabel('Angular Frequency ω (rad/s)', fontsize=label_fontsize)
ax_mag.set_ylabel('|H(jω)|', fontsize=label_fontsize)
ax_mag.set_title('Butterworth Magnitude Response', fontsize=title_fontsize, fontweight='bold', pad=5)
ax_mag.tick_params(axis='both', labelsize=tick_fontsize)
ax_mag.grid(True, which='both', linestyle=':', alpha=0.5)
ax_mag.legend(loc='upper center', bbox_to_anchor=(0.5, -0.23), ncol=3, fontsize=legend_fontsize)

ax_mag.set_xlim(0.01, 100.0)
ax_mag.set_ylim(0.0, 1.08)

fig_mag.subplots_adjust(left=0.14, right=0.97, bottom=0.30, top=0.85)
fig_mag.canvas.header_visible = False
fig_mag.canvas.toolbar_visible = False
fig_mag.canvas.resizable = False
fig_mag.canvas.layout.width = '530px'
fig_mag.canvas.layout.height = '305px'

# ==============================================================================
# FIGURE 2: PHASE RESPONSE
# ==============================================================================

fig_phase, ax_phase = plt.subplots(figsize=(5.3, 3.0))

phase_line, = ax_phase.plot(omega_values, phase_deg_values, 'r-', linewidth=2.0, label='∠H(jω)')

ax_phase.axvline(1.0, color='black', linestyle=':', linewidth=1.0, label='ω = 1')
ax_phase.axhline(0.0, color='gray', linestyle='--', linewidth=0.8)

ax_phase.set_xscale('log')
ax_phase.set_xlabel('Angular Frequency ω (rad/s)', fontsize=label_fontsize)
ax_phase.set_ylabel('Phase (degrees)', fontsize=label_fontsize)
ax_phase.set_title('Butterworth Phase Response', fontsize=title_fontsize, fontweight='bold', pad=5)
ax_phase.tick_params(axis='both', labelsize=tick_fontsize)
ax_phase.grid(True, which='both', linestyle=':', alpha=0.5)
ax_phase.legend(loc='upper center', bbox_to_anchor=(0.5, -0.23), ncol=2, fontsize=legend_fontsize)

ax_phase.set_xlim(0.01, 100.0)
ax_phase.set_ylim(-365.0, 5.0)
ax_phase.set_yticks([0, -45, -90, -135, -180, -225, -270, -315, -360])

fig_phase.subplots_adjust(left=0.14, right=0.97, bottom=0.30, top=0.85)
fig_phase.canvas.header_visible = False
fig_phase.canvas.toolbar_visible = False
fig_phase.canvas.resizable = False
fig_phase.canvas.layout.width = '530px'
fig_phase.canvas.layout.height = '305px'

# ==============================================================================
# FIGURE 3: GROUP DELAY
# ==============================================================================

fig_gd, ax_gd = plt.subplots(figsize=(5.3, 3.0))

gd_line, = ax_gd.plot(omega_values, group_delay_values, 'r-', linewidth=2.0, label='τ(ω)')

ax_gd.axvline(1.0, color='black', linestyle=':', linewidth=1.0, label='ω = 1')
ax_gd.axhline(0.0, color='gray', linestyle='--', linewidth=0.8)

ax_gd.set_xscale('log')
ax_gd.set_xlabel('Angular Frequency ω (rad/s)', fontsize=label_fontsize)
ax_gd.set_ylabel('Group Delay τ(ω)', fontsize=label_fontsize)
ax_gd.set_title('Butterworth Group Delay', fontsize=title_fontsize, fontweight='bold', pad=5)
ax_gd.tick_params(axis='both', labelsize=tick_fontsize)
ax_gd.grid(True, which='both', linestyle=':', alpha=0.5)
ax_gd.legend(loc='upper center', bbox_to_anchor=(0.5, -0.23), ncol=2, fontsize=legend_fontsize)

ax_gd.set_xlim(0.01, 100.0)
ax_gd.set_ylim(0.0, 4.0)
ax_gd.set_yticks(np.arange(0.0, 4.1, 0.5))

fig_gd.subplots_adjust(left=0.14, right=0.97, bottom=0.30, top=0.85)
fig_gd.canvas.header_visible = False
fig_gd.canvas.toolbar_visible = False
fig_gd.canvas.resizable = False
fig_gd.canvas.layout.width = '530px'
fig_gd.canvas.layout.height = '305px'

# ==============================================================================
# LAYOUT
# ==============================================================================

left_column = VBox([info_html], layout=Layout(width='550px', min_width='550px', max_width='550px', flex='0 0 550px', align_items='flex-start'))

right_column = VBox([fig_mag.canvas, fig_phase.canvas, fig_gd.canvas], layout=Layout(width='540px', min_width='540px', max_width='540px', flex='0 0 540px', align_items='flex-start'))

main_layout = HBox([left_column, right_column], layout=Layout(width='1100px', min_width='1100px', max_width='1100px', align_items='flex-start', justify_content='flex-start'))

# ==============================================================================
# DISPLAY
# ==============================================================================

display(description)
display(main_layout)